# 03 -- Typer

## Objectifs pedagogiques

A la fin de ce notebook, vous saurez :

- creer des CLI modernes avec Typer ;
- utiliser les type hints pour les arguments et options ;
- gerer les sous-commandes et les groupes ;
- ajouter de l'aide automatique et des completions shell ;
- valider les entrees avec `Annotated` et des callbacks ;
- utiliser les prompts interactifs ;
- tester les CLI avec `CliRunner`.

## Prerequis -- ce que vous devez maitriser

- les fonctions typees et les type hints avances (`Annotated`, `Optional`) ;
- la notion de CLI et de code de retour ;

Ce que ce notebook **ne couvrira pas** :

- Rich pour la sortie formatee (voir `04_Rich_console.ipynb`) ;
- Click en detail (Typer est base sur Click).

## Plan

1. Installation et premier CLI
2. Arguments et Options
3. Types supportes
4. Options avec Annotated
5. Sous-commandes
6. Prompts interactifs
7. Callbacks et validation
8. Couleurs et formatage
9. Tester avec CliRunner
10. Synthese
11. Exercices

---

## 1. Installation et premier CLI

In [ ]:
# pip install typer
import typer

### Le CLI le plus simple

In [ ]:
# Dans un fichier hello.py :
# import typer
#
# def main(name: str) -> None:
#     print(f'Hello {name}')
#
# if __name__ == '__main__':
#     typer.run(main)

# $ python hello.py World
# Hello World
print('Voir le code dans une cellule markdown ci-dessus')

Typer utilise les **type hints** de la fonction pour generer automatiquement le parsing des arguments.

---

## 2. Arguments et Options

- **Argument** = parametre positionnel (obligatoire par defaut)
- **Option** = parametre nomme avec `--` (optionnel par defaut)

In [ ]:
from typing import Annotated, Optional

app = typer.Typer()

@app.command()
def greet(
    name: str,  # argument positionnel
    greeting: str = 'Hello',  # option avec valeur par defaut
    loud: bool = False,  # flag booleen
) -> None:
    msg = f'{greeting} {name}'
    if loud:
        msg = msg.upper()
    print(msg)

# En CLI : python script.py World --greeting Salut --loud

---

## 3. Types supportes

Typer supporte nativement : `str`, `int`, `float`, `bool`, `Path`, `datetime`, `Enum`, `list[str]`, `tuple`.

In [ ]:
from pathlib import Path
from enum import Enum

class Color(str, Enum):
    red = 'red'
    green = 'green'
    blue = 'blue'

@app.command('colored')
def colored_output(
    text: str,
    color: Color = Color.green,
    count: int = 1,
    output: Path = Path('/tmp/output.txt'),
) -> None:
    for _ in range(count):
        print(f'[{color.value}] {text}')

---

## 4. Options avec Annotated

`Annotated` permet de configurer l'aide, les noms courts, et la validation.

In [ ]:
@app.command('search')
def search(
    query: Annotated[str, typer.Argument(help='Terme de recherche')],
    max_results: Annotated[int, typer.Option('--max', '-m', help='Nombre max de resultats')] = 10,
    verbose: Annotated[bool, typer.Option('--verbose', '-v', help='Mode verbeux')] = False,
) -> None:
    print(f'Recherche de {query!r}, max={max_results}, verbose={verbose}')

---

## 5. Sous-commandes

Typer gere les sous-commandes comme Git (`git add`, `git commit`).

In [ ]:
users_app = typer.Typer(help='Gestion des utilisateurs')
app.add_typer(users_app, name='users')

@users_app.command('add')
def add_user(name: str, email: str) -> None:
    print(f'Utilisateur {name} ({email}) ajoute')

@users_app.command('list')
def list_users() -> None:
    print('Liste des utilisateurs')

---

## 6. Prompts interactifs

In [ ]:
# @app.command()
# def login(
#     username: str = typer.Option(..., prompt=True),
#     password: str = typer.Option(..., prompt=True, hide_input=True),
# ) -> None:
#     print(f'Login as {username}')

# Confirmation
# delete = typer.confirm('Supprimer ?')
print('Prompts disponibles en mode interactif')

---

## 7. Callbacks et validation

In [ ]:
def validate_port(value: int) -> int:
    if value < 1 or value > 65535:
        raise typer.BadParameter('Le port doit etre entre 1 et 65535')
    return value

@app.command('serve')
def serve(
    port: Annotated[int, typer.Option(callback=validate_port)] = 8000,
) -> None:
    print(f'Serveur sur le port {port}')

---

## 8. Couleurs et formatage

In [ ]:
# typer.echo utilise les couleurs ANSI
# typer.echo(typer.style('Succes', fg=typer.colors.GREEN, bold=True))
# typer.echo(typer.style('Erreur', fg=typer.colors.RED))
print('Couleurs disponibles : RED, GREEN, BLUE, YELLOW, MAGENTA, CYAN, WHITE')

---

## 9. Tester avec CliRunner

In [ ]:
from typer.testing import CliRunner

runner = CliRunner()

# Test d'une commande
test_app = typer.Typer()

@test_app.command()
def hello(name: str) -> None:
    print(f'Hello {name}')

result = runner.invoke(test_app, ['World'])
result.exit_code

In [ ]:
result.output.strip()

---

## Synthese

| Concept | Typer |
|---|---|
| Argument positionnel | `name: str` |
| Option avec defaut | `name: str = 'default'` |
| Flag booleen | `verbose: bool = False` (genere `--verbose` / `--no-verbose`) |
| Sous-commandes | `app.add_typer(sub_app, name='sub')` |
| Validation | `callback=` dans `typer.Option` |
| Prompt | `typer.Option(prompt=True)` |
| Test | `CliRunner().invoke(app, args)` |

### Regles a retenir

- **Type hints = parsing** : Typer deduit tout des annotations.
- **`Annotated`** pour la configuration avancee (aide, noms courts).
- **Sous-commandes** pour les CLI complexes.
- **`CliRunner`** pour les tests automatises.
- **Enum** pour les choix valides.

---

## Exercices

### Exercice 1 -- CLI de gestion de taches

Creez une application Typer avec 3 sous-commandes : `add <titre>`, `list`, `done <id>`. Stockez les taches en memoire (dict).

In [ ]:
import typer

# TODO

<details>
<summary>Voir la correction</summary>

```python
import typer

app = typer.Typer()
tasks: dict[int, dict] = {}
next_id = 1

@app.command()
def add(titre: str) -> None:
    global next_id
    tasks[next_id] = {'titre': titre, 'done': False}
    print(f'Tache #{next_id} ajoutee')
    next_id += 1

@app.command('list')
def list_tasks() -> None:
    for tid, task in tasks.items():
        status = 'X' if task['done'] else ' '
        print(f'[{status}] #{tid}: {task["titre"]}')

@app.command()
def done(task_id: int) -> None:
    if task_id in tasks:
        tasks[task_id]['done'] = True
        print(f'Tache #{task_id} terminee')
    else:
        print(f'Tache #{task_id} introuvable')
```

</details>

### Exercice 2 -- CLI avec validation

Creez une commande `convert` qui prend un fichier CSV en entree et un format de sortie (`json` ou `yaml`). Validez que le fichier existe et que le format est supporte.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
import typer
from pathlib import Path
from enum import Enum
from typing import Annotated

class Format(str, Enum):
    json = 'json'
    yaml = 'yaml'

app = typer.Typer()

@app.command()
def convert(
    input_file: Annotated[Path, typer.Argument(exists=True, help='Fichier CSV')],
    output_format: Annotated[Format, typer.Option('--format', '-f')] = Format.json,
) -> None:
    print(f'Conversion de {input_file} en {output_format.value}')
```

</details>

---

## Ressources externes

- [Documentation Typer](https://typer.tiangolo.com/)
- [Typer Tutorial](https://typer.tiangolo.com/tutorial/)
- [Click (base de Typer)](https://click.palletsprojects.com/)